In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

# Пути к данным M5
data_dir = "../data/m5"
sales = pd.read_csv(f"{data_dir}/sales_train_evaluation.csv")
calendar = pd.read_csv(f"{data_dir}/calendar.csv")
prices = pd.read_csv(f"{data_dir}/sell_prices.csv")

# Для примера берём один товар и один магазин
store_id = "CA_1"
item_id = "HOBBIES_1_001"

df = sales[(sales["store_id"] == store_id) & (sales["item_id"] == item_id)].copy()

# Широкий формат -> длинный
day_cols = [c for c in df.columns if c.startswith("d_")]
df_long = df.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# Добавляем календарь
df_long = df_long.merge(
    calendar[["d", "date", "wm_yr_wk", "weekday", "wday", "month", "year"]],
    on="d", how="left"
)
df_long["date"] = pd.to_datetime(df_long["date"])
df_long = df_long.sort_values("date")

# Лаговые признаки
for lag in [1, 7, 14, 28]:
    df_long[f"lag_{lag}"] = df_long["sales"].shift(lag)

df_long["rolling_mean_7"] = df_long["sales"].shift(1).rolling(7).mean()
df_long["rolling_mean_28"] = df_long["sales"].shift(1).rolling(28).mean()
df_long["day_of_week"] = df_long["date"].dt.dayofweek
df_long["month"] = df_long["date"].dt.month

df_model = df_long.dropna().copy()

features = ["lag_1", "lag_7", "lag_14", "lag_28",
            "rolling_mean_7", "rolling_mean_28", "day_of_week", "month"]
target = "sales"

# Разделение: последние 28 дней — тест
last_date = df_model["date"].max()
train = df_model[df_model["date"] < last_date - pd.Timedelta(days=28)]
test = df_model[df_model["date"] >= last_date - pd.Timedelta(days=28)]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Модель
model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

# Оценка
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

# Рекурсивный прогноз на 28 дней вперёд
history = df_model[["date", "sales"]].copy()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=28, freq="D")
forecasts = []

for date in forecast_dates:
    row = {}
    for lag in [1, 7, 14, 28]:
        row[f"lag_{lag}"] = history["sales"].iloc[-lag]
    row["rolling_mean_7"] = history["sales"].iloc[-7:].mean()
    row["rolling_mean_28"] = history["sales"].iloc[-28:].mean()
    row["day_of_week"] = date.dayofweek
    row["month"] = date.month
    X_pred = pd.DataFrame([row])[features]
    pred = model.predict(X_pred)[0]
    forecasts.append({"date": date, "sales": pred})
    history = pd.concat([
        history,
        pd.DataFrame({"date": [date], "sales": [pred]})
    ], ignore_index=True)

forecast_df = pd.DataFrame(forecasts)
print(forecast_df.head())

# Сохранение
os.makedirs("../models", exist_ok=True)
joblib.dump({
    "model": model,
    "features": features,
    "store_id": store_id,
    "item_id": item_id
}, "../models/walmart_lgbm.joblib")
print("Модель сохранена: ../models/walmart_lgbm.joblib")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001462 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 90
[LightGBM] [Info] Number of data points in the train set: 1884, number of used features: 8
[LightGBM] [Info] Start training from score 0.317941
MAE: 1.13, RMSE: 1.51
        date     sales
0 2016-05-23  1.276066
1 2016-05-24  1.229301
2 2016-05-25  1.148526
3 2016-05-26 -0.129196
4 2016-05-27  0.820331
Модель сохранена: ../models/walmart_lgbm.joblib
